# 11 — Trade Card Generation

Notebook này tạo các Trade Card cho các tín hiệu mua (breakout) và bán (pullback, volume spike).

In [3]:
# Cell: Setup Project Root
import os
import sys
from pathlib import Path


def find_project_root(start_path=None):
    path = Path(start_path or os.getcwd()).resolve()
    for parent in [path, *path.parents]:
        if (parent / "src").is_dir() and (parent / "config").is_dir():
            return parent
    raise FileNotFoundError("Khong tim thay project root co thu muc src/ va config/")


project_root = find_project_root()
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root:", project_root)


Project root: /Users/jamesb/Desktop/vietnam-quant-research


In [4]:
# Cell: Import Trade Card Pipeline Helpers
from pathlib import Path

import pandas as pd

from src.data.loaders import read_parquet
from src.reporting.trade_card_pipeline import build_daily_watchlist, build_trade_cards, save_trade_cards
from src.screening.light_market_scan import load_auto_watchlist_symbols


In [5]:
# Cell: Load Inputs And Clean Features
features_path = Path("data/signals/setup_signals.parquet")
stock_features_path = Path("data/processed/stock_features.parquet")
eligibility_path = Path("data/signals/breakout_20d_setup_eligibility.parquet")
vol_rule_path = Path("data/signals/breakout_20d_volatility_rule.parquet")

features = read_parquet(features_path)
stock_features = read_parquet(stock_features_path)
eligibility = read_parquet(eligibility_path)
vol_rule = read_parquet(vol_rule_path)

if not features.empty:
    features["date"] = pd.to_datetime(features["date"])
if not stock_features.empty and "date" in stock_features.columns:
    stock_features["date"] = pd.to_datetime(stock_features["date"])

# Enrich setup signals with volatility regime if stock features exist.
if "volatility_regime" not in features.columns and not stock_features.empty:
    enrich_cols = [
        col
        for col in ["symbol", "date", "volatility_regime", "volatility_percentile", "atr_pct_14"]
        if col in stock_features.columns
    ]
    if {"symbol", "date", "volatility_regime"}.issubset(enrich_cols):
        features = features.merge(
            stock_features[enrich_cols].drop_duplicates(subset=["symbol", "date"]),
            on=["symbol", "date"],
            how="left",
        )

print({
    "features_rows": len(features),
    "unique_symbols": int(features["symbol"].nunique()) if "symbol" in features.columns else 0,
    "has_volatility_regime": "volatility_regime" in features.columns,
})


{'features_rows': 178493, 'unique_symbols': 499, 'has_volatility_regime': True}


In [6]:
# Cell: Verify Signals & Eligibility
feature_symbol_set = set(features["symbol"].astype(str).str.upper()) if not features.empty and "symbol" in features.columns else set()
latest_view = features.sort_values("date").groupby("symbol").tail(1) if not features.empty else pd.DataFrame()
latest_symbol_set = set(latest_view["symbol"].astype(str).str.upper()) if not latest_view.empty else set()
eligibility_symbol_set = set(eligibility["symbol"].astype(str).str.upper()) if not eligibility.empty and "symbol" in eligibility.columns else set()

diagnostic = {
    "total_symbols_in_dataset": len(feature_symbol_set),
    "latest_symbols": len(latest_symbol_set),
    "symbols_with_breakout": int(latest_view.get("breakout_20d", pd.Series(dtype=bool)).fillna(False).sum()) if not latest_view.empty else 0,
    "symbols_with_vol_regime": int(latest_view["volatility_regime"].notna().sum()) if "volatility_regime" in latest_view.columns else 0,
    "eligibility_rows": len(eligibility),
    "vol_rule_rows": len(vol_rule),
}
print(diagnostic)
if not latest_view.empty:
    display(latest_view[[col for col in ["symbol", "date", "close", "breakout_20d", "volatility_regime", "atr_pct_14"] if col in latest_view.columns]].head(10))


{'total_symbols_in_dataset': 499, 'latest_symbols': 499, 'symbols_with_breakout': 2, 'symbols_with_vol_regime': 499, 'eligibility_rows': 481, 'vol_rule_rows': 6}


,symbol,date,close,breakout_20d,volatility_regime,atr_pct_14
16627,ATB,2025-05-30,600,False,Unknown,NaN
14942,ARM,2026-03-16,45798,False,Unknown,0.032202
51496,DTT,2026-06-03,17650,False,Unknown,NaN
11151,AME,2026-06-15,6800,False,Unknown,0.047269
33889,COM,2026-06-15,31500,False,Unknown,0.043651
132657,SPM,2026-06-15,8160,False,Unknown,0.037990
77893,HU1,2026-06-16,5700,False,Unknown,0.052632
145789,TIX,2026-06-16,38400,False,Unknown,0.049206
29469,CCI,2026-06-17,19250,False,Unknown,0.036549
154428,VAF,2026-06-18,26000,False,Unknown,0.069780


In [7]:
# Cell: Build Trade Cards
trade_cards = build_trade_cards(
    features=features,
    eligibility=eligibility,
    vol_rule=vol_rule,
    signal_col="breakout_20d",
    latest_only=True,
)
trade_cards


,symbol,date,close,breakout_20d,edge_tier,eligible,watchlist_eligible,volatility_regime,volatility_rule,profit_factor,avg_return,median_return,stop_loss,holding_period,setup_quality_score,trade_edge_score,action,data_is_saying
0,ASP,2026-06-29,8910,True,Conditional Edge,False,True,Stress Vol,Allowed,1.564901,0.025421,-0.054000,-0.05,20.0,75,75,Watch - Conditional Breakout,ASP: có tín hiệu breakout nhưng điều kiện chưa...
1,VCG,2026-06-29,20950,False,Emerging Edge,False,True,Normal Vol,Allowed,55.695794,0.177627,0.116516,-0.05,20.0,76,0,Watchlist - Emerging,VCG: thống kê breakout trên mã này còn ít mẫu ...
2,CTS,2026-06-29,25100,False,Valid Edge,True,True,High Vol,Allowed,1.709146,0.023402,-0.054000,-0.05,20.0,74,0,Watchlist - No Signal,"CTS: chưa có breakout 20D mới, nhưng mã này nằ..."
3,MSB,2026-06-29,16000,False,Emerging Edge,False,True,Normal Vol,Allowed,46.730032,0.308678,0.267843,-0.05,20.0,72,0,Watchlist - Emerging,MSB: thống kê breakout trên mã này còn ít mẫu ...
4,OCB,2026-06-29,12450,False,Emerging Edge,False,True,High Vol,Allowed,37.683407,0.440201,0.459145,-0.05,20.0,72,0,Watchlist - Emerging,OCB: thống kê breakout trên mã này còn ít mẫu ...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
494,GDT,2026-06-23,16500,False,Insufficient Data,False,False,Panic Vol,Avoid,0.000000,-0.049761,-0.054000,-0.05,20.0,0,0,No Signal,GDT: chưa có breakout 20D mới và edge lịch sử ...
495,ST8,2026-06-24,3080,False,Insufficient Data,False,False,Panic Vol,Avoid,0.000000,-0.054000,-0.054000,-0.05,20.0,0,0,No Signal,ST8: chưa có breakout 20D mới và edge lịch sử ...
496,VPD,2026-06-23,21950,False,Insufficient Data,False,False,Panic Vol,Avoid,0.000000,-0.039499,-0.039499,-0.05,20.0,0,0,No Signal,VPD: chưa có breakout 20D mới và edge lịch sử ...
497,VTO,2026-06-24,10500,False,Insufficient Data,False,False,Panic Vol,Avoid,0.000000,-0.037344,-0.054000,-0.05,20.0,0,0,No Signal,VTO: chưa có breakout 20D mới và edge lịch sử ...


In [8]:
# Cell: Build Daily Watchlist
daily_watchlist = build_daily_watchlist(trade_cards, signal_col="breakout_20d")
daily_watchlist


,symbol,date,close,breakout_20d,edge_tier,volatility_regime,volatility_rule,setup_quality_score,trade_edge_score,action
0,ASP,2026-06-29,8910,True,Conditional Edge,Stress Vol,Allowed,75,75,Watch - Conditional Breakout
1,LAF,2026-06-29,22200,True,Insufficient Data,Unknown,Ignore,35,0,Watch - Need Confirmation
2,CTS,2026-06-29,25100,False,Valid Edge,High Vol,Allowed,74,0,Watchlist - No Signal
3,HDB,2026-06-29,25750,False,Valid Edge,Low Vol,Allowed,67,0,Watchlist - No Signal
4,CTG,2026-06-29,33950,False,Valid Edge,Low Vol,Allowed,67,0,Watchlist - No Signal
5,NAF,2026-06-29,49950,False,Valid Edge,Low Vol,Allowed,66,0,Watchlist - No Signal
6,BFC,2026-06-29,56100,False,Valid Edge,Low Vol,Allowed,66,0,Watchlist - No Signal
7,SZC,2026-06-29,22350,False,Valid Edge,Low Vol,Allowed,60,0,Watchlist - No Signal
8,DIG,2026-06-29,12350,False,Valid Edge,Low Vol,Allowed,60,0,Watchlist - No Signal
9,VCG,2026-06-29,20950,False,Emerging Edge,Normal Vol,Allowed,76,0,Watchlist - Emerging


In [9]:
# Cell: Save Trade Cards
save_trade_cards(trade_cards, path="data/signals/breakout_20d_trade_cards.parquet")
print("Saved trade cards to data/signals/")


Saved trade cards to data/signals/
